In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col

In [0]:
# Define table names using environment variables
silver_races_table = f"{catalog_name}.{silver_schema}.races"
silver_circuits_table = f"{catalog_name}.{silver_schema}.circuits"
gold_dim_races_table = f"{catalog_name}.{gold_schema}.dim_races"

# Read silver races table
races_df = spark.table(silver_races_table)

# Read silver circuits table
circuits_df = spark.table(silver_circuits_table)

In [0]:
# Join races and circuits on circuit_id
dim_races_df = (
    races_df.alias("races")
    .join(
        circuits_df.alias("circuits"),
        col("races.circuit_id") == col("circuits.circuit_id"),
        "inner"
    )
    # Select only required columns
    .select(
        col("races.season"),
        col("races.round"),
        col("races.race_name"),
        col("races.race_date"),
        col("circuits.circuit_name"),
        col("circuits.locality"),
        col("circuits.country")
    )
)

In [0]:
# Write transformed data to gold layer
(
    dim_races_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(gold_dim_races_table)
)

print(f"✓ Races dimension successfully written to {gold_dim_races_table}")

In [0]:
# Preview the races dimension table
#spark.table(gold_dim_races_table).limit(10).display()